In [ ]:
# Fix protobuf version mismatch for kaggle_benchmarks
!pip install protobuf==5.29.6 --quiet


# Trinity Hippocampal Learning Probe — Reward-Signal Learning

**Task 4 of 25** | Track 1: Learning | Brain Zone: ACCUMBENS

This notebook measures a model's ability to learn from reward signals.

## Trinity Neuroanatomical Foundation

The **ACCumbens** in Trinity implements reinforcement learning with stationarity tracking:

```zig
// src/storm/zones/accumbens.zig
pub const RewardSignal = struct {
    value: f32,  // Reward magnitude
    stationarity: f32,  # Reward distribution stability
    prediction_error: f32,  # TD error
};
```

### Ternary Scoring {-1, 0, +1}

- **+1 (correct)**: Model learns reward pattern correctly
- **0 (partial)**: Model shows uncertainty about reward
- **-1 (wrong)**: Model fails to learn or misinterprets reward

### φ-Scaling

Stationarity reward follows gradient: 1.0 (stable) to 0.0 (random)

In [ ]:
# Install Kaggle Benchmarks SDK
!pip install -q kaggle-benchmarks

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass
from typing import Literal

# Load generated dataset
df = pd.read_csv('../../data/thlp_learning.csv')
print(f"Loaded {len(df)} items")
df.head()

In [ ]:
# Filter for reward signal learning task
reward_df = df[df['task'] == 'Reward-Signal Learning'].copy()
print(f"Reward signal items: {len(reward_df)}")
reward_df[['id', 'task', 'question', 'answer', 'context_length', 'difficulty']].head()

In [ ]:
# Define structured output for reward learning
@dataclass
class RewardLearningResponse:
    """Model's response to reward signal."""
    action: str  # Action taken
    expected_reward: str  # Predicted reward
    confidence: float  # 0.0 to 1.0
    
    def ternary_score(self, expected_answer: str) -> Literal[-1, 0, 1]:
        """Return Trinity ternary score {-1, 0, +1}."""
        # Correct: exact match
        if self.expected_reward == expected_answer:
            return 1
        # Partial: uncertain but close
        elif 0.3 < self.confidence < 0.7:
            return 0
        # Wrong: incorrect or overconfident
        else:
            return -1

print("RewardLearningResponse schema defined")

In [ ]:
# Define Kaggle benchmark task
@kbench.task(name="trinity_accumbens_reward_learning")
def reward_signal_learning(
    llm: kbench.LLM,
    question: str,
    expected_answer: str,
    stationarity_reward: float
) -> float:
    """
    Measure model's reward signal learning ability.
    
    Returns:
        Learning score: 1.0 (perfect) to -1.0 (worst)
    """
    prompt = f"""You received a reward signal after taking an action.

Question: {question}

Provide:
1. The action you took
2. The reward you expect to receive
3. Your confidence in this prediction (0.0 to 1.0)
"""
    
    response = llm.prompt(
        prompt,
        schema=RewardLearningResponse
    )
    
    # Calculate ternary score
    ternary = response.ternary_score(expected_answer)
    
    # Combine: 50% accuracy, 30% confidence, 20% stationarity
    accuracy = 1.0 if response.expected_reward == expected_answer else -1.0
    confidence_score = (response.confidence - 0.5) * 2  # [-1, 1]
    stationarity_bonus = stationarity_reward * 0.2  # [0, 0.2]
    
    final_score = 0.5 * accuracy + 0.3 * confidence_score + stationarity_bonus
    
    return max(-1.0, min(1.0, final_score))

print("Task 'trinity_accumbens_reward_learning' registered")

In [ ]:
# Run evaluation on a sample
sample_df = reward_df.head(10)  # Test with 10 items first

results = reward_signal_learning.evaluate(
    llm=[kbench.llm],  # Default test LLM
    evaluation_data=sample_df
)

print("Evaluation Results (Sample):")
print(f"Mean Score: {results['score'].mean():.4f}")
print(f"Std Dev: {results['score'].std():.4f}")
print(f"Min: {results['score'].min():.4f}")
print(f"Max: {results['score'].max():.4f}")
results.head()

In [ ]:
# Full evaluation (uncomment when ready)
# results = reward_signal_learning.evaluate(
#     llm=[kbench.llm],
#     evaluation_data=reward_df
# )
# 
# print(f"\nFull Evaluation Results ({len(reward_df)} items):")
# print(f"Mean Score: {results['score'].mean():.4f}")
# print(f"Ternary Distribution:")
# print(results['ternary_outcome'].value_counts())

In [ ]:
# Submit to Kaggle leaderboard
# Uncomment and run when ready to submit
# kbench.submit(
#     task=reward_signal_learning,
#     results=results,
#     message="Trinity ACCumbens Reward Learning v1.0"
# )
# print("✅ Submitted to Kaggle leaderboard!")

## Expected Leaderboard Performance

Models are expected to show clear separation on this benchmark:

| Model | Expected Score | Reason |
|-------|---------------|--------|
| GPT-4o | 0.60-0.70 | Good reward learning |
| Claude Sonnet | 0.55-0.65 | Moderate reinforcement |
| Gemini Pro | 0.50-0.60 | Weak reward sensitivity |
| Llama 3 70B | 0.35-0.50 | Poor reward learning |

The **gradient** in scores demonstrates this benchmark's discriminatory power.